# The behaviour screen

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/qwen-symbolic-thinking/blob/main/notebooks/screen.ipynb)

Can a Qwen3.5 vision-language model, shown a picture and asked plainly, think it through in writing the way the rows of
[thought-symbolic-image-captions](https://huggingface.co/datasets/AbstractPhil/thought-symbolic-image-captions) do?
Three prompt forms (bare instructions; two worked examples shown; a specification instead of a picture), two models, about
a hundred rows each. Results ship to [qwen-symbolic-thinking-image-training](https://huggingface.co/AbstractPhil/qwen-symbolic-thinking-image-training)
under `screens/<stamp>/`.

Runtime: a GPU (the RTX PRO 6000 is the reference card). Secrets: `HF_TOKEN` with notebook access, so the results can ship.

Run C1, then **Runtime > Restart session**, then C2 (the checks), C3 (the screen) and C4 (the throughput sweep) as wanted, each by its own run button.

In [ ]:
# C1: install, then Runtime > Restart session
import importlib.util, subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("git+https://github.com/AbstractEyes/qwen-symbolic-thinking@v0.1.5", "transformers>=5.0", "accelerate>=1.0", "peft>=0.18")

FAST_PATH = True   # the linear-attention kernels Qwen3.5 uses; causal-conv1d compiles for about 5 minutes; all or nothing
if FAST_PATH:
    if all(importlib.util.find_spec(m) for m in ("causal_conv1d", "fla", "triton")):
        print("fast path already present")
    else:
        try:
            pip("causal-conv1d", "--no-build-isolation")
            pip("flash-linear-attention", "triton")
            print("fast path on")
        except subprocess.CalledProcessError as e:
            print("fast path off (the screen still runs, slower):", e)
            subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "causal-conv1d"])

try:                # Colab's torchao 0.10 breaks PEFT when it builds a LoRA (PEFT wants 0.16 or newer, or none at all)
    import torchao
    _v = tuple(int(x) for x in torchao.__version__.split(".")[:2])
    if _v < (0, 16):
        subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
        print("torchao", torchao.__version__, "removed: PEFT cannot build a LoRA beside it")
    else:
        print("torchao", torchao.__version__, "stays")
except ImportError:
    print("no torchao")

print("installed; now Runtime > Restart session, then run C2")

In [ ]:
# C2: the checks (seconds): the linear-attention kernels present? does PEFT build a LoRA beside the installed torchao?
from qwen_symbolic_thinking.checks import run as checks

report = checks()

In [ ]:
# C3: the screen (about 100 rows x 3 forms x 2 models; progress lines show the time left)
from qwen_symbolic_thinking.screen import run

summary = run(models=["Qwen/Qwen3.5-9B", "Qwen/Qwen3.5-4B"],
              inputs="hub:screens/inputs/screen_v1.json",
              batch=4, max_new_tokens=700)

In [ ]:
# C4: the throughput sweep (the same 60 picture rows at each batch size; an out-of-memory batch ends it)
from qwen_symbolic_thinking.screen import throughput

sweep = throughput("Qwen/Qwen3.5-9B", inputs="hub:screens/inputs/screen_v1.json",
                   batches=(4, 8, 16, 32, 64), n_rows=64, form="shown", max_new_tokens=700)

In [ ]:
# C5: the big-batch point (256 picture rows of the fine-tune set, the one-line form, 400 new tokens at most)
from qwen_symbolic_thinking.screen import throughput

big = throughput("Qwen/Qwen3.5-9B", inputs="hub:sft/sft_v1.json", batches=(64, 128, 256), n_rows=256,
                 form="short", max_new_tokens=400, tag="throughput_big")